# Notebook 1 <br> Pandas basics: structures, loading, selecting, seeing

## Introduction

When processing data, you will most often work with _structured data_. This is the kind of data where you have a table with some number of named columns, each corresponding to a different feature, and a sequence of rows, each corresponding to a different sample. Examples of structured data can be found everywhere and occur any time you are making a series of observations and write down some properties for each observation you make.

Some examples of structured data are:
- Weather reports, where you have columns such as 'Date', 'Temperature' and 'Humidity', and each row shows the temperature and humidity on some date.
- Companies use structured data everywhere, from user accounts to product information and from purchases to reviews.
- A grocery list is a very simple type of structured data, where you have a 'Product' and 'Amount' column indicating how many of what product you intend to buy.

_Note on notation:_ There are many different ways to refer to the columns and rows in a table. This notebook will refer to the vertically orientated axis as 'columns' (but elsewhere this might be called a 'field', 'feature' or 'property') and the horizontal axis as 'rows' (also called 'entries', 'samples' or 'observations').

`Pandas` (often imported as `pd`) is the most popular library to use when dealing with structured data in Python. In this exercise notebook, we will explore some of the basics of working with Pandas. This notebook will not be able to cover everything you will ever need, but will hopefully give you an idea of what it can do for you. 

Some sections will end with a note on *Further Reading*; this will not be necessary for completing the exercises, but references relevant information that might help you in the future.

## Chapter 1: Series & DataFrames

<!-- **What this chapter teaches**

- What a DataFrame is (a table = aligned columns) and what a Series is (one column / a 1-D labelled array)
- Creating DataFrames and Series from dicts and lists
- The index: row labels vs. positions
- Accessing a column with `df["col"]` and seeing that a column *is* a Series
- Adding, modifying, and deleting columns and rows


> _Reused material from the existing `pandas.ipynb` follows below. Still to add: the validation checkpoint, and heading/level cleanup._ -->
Before we start, load the Pandas package below.

In [ ]:
import pandas as pd
import pooch

%reload_ext autoreload
%autoreload 2

pd.set_option("display.precision", 2)

This notebook needs a few extra files: the data you will work with, and `tests.py`, which checks your answers. Run the cell below to download them into the folder this notebook is in. You only need to do this once; after that the files are already on your computer and the cell finishes immediately.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/spcourse/pandas/main/pandas-dp/data/"

print("downloading data files")
for fname in ["products.csv", "survey_2022.csv", "survey_2023.csv", "tests.py"]:
    pooch.retrieve(url=DATA_URL + fname, known_hash=None, fname=fname, path=".", progressbar=True)
print("done!")

from tests import *

### DataFrames

Structured data is nothing more than a table consisting of rows and columns. In `Pandas`, such a table is called a `DataFrame`. To create a DataFrame called `df`, you can use `df = pd.DataFrame(data)` where `data` is a `list` of rows, and each row is also a list, containing the values for that row. Below you can see a simple example of creating a table with 2 rows and 3 columns.

You can have any number of rows, but it is important that each row has the same number of elements (i.e. columns), otherwise you will get an error. The example below uses Jupyter's `display` function to get a nice looking output (although `print` also works).

In [ ]:
data = [[1, 2, 3], # The first row has values 1, 2, 3
        [4, 5, 6]] # The second row has values 4, 5, 6

df = pd.DataFrame(data)

display(df)

Now this is not a very descriptive table, since the columns are just called `0`, `1` and `2`. To change this, you can use the optional `columns` argument when creating the `DataFrame`, where you can specify the names of each of the columns using a `list`. Make sure to provide as many column names as there are columns in your data (the number of values in a single row), otherwise you will get an error.

Let's create another dataframe; the first column has values `'a'` and `'b'` and we call it `'Some letter'`, another column with values `2` and `5` we call `'Int'` and finally a column with `3.5` and `6.2` we call `'A float'`.

In [ ]:
data = [['a', 2, 3.5],
        ['b', 5, 6.2]]

df = pd.DataFrame(data, columns=['Some letter', 'Int', 'A float'])

display(df)

You might have already noticed that each of the _rows_ also gets its own index on the left (in this case `0` and `1`, which Pandas assigns by default). This is called the `index` of the dataframe (which you can see by using `df.index`), and can be set using the optional `index` argument to provide a `list` of row names when creating the `DataFrame`.

Again, make sure you provide as many names in the `index` argument as there are rows (in this case 2 rows, and therefore 2 names); otherwise you will get an error.

In [ ]:
data = [['a', 2, 3.5],
        ['b', 5, 6.2]]

df = pd.DataFrame(data,
    index=['First Row', 'Row 2'],
    columns=['Some letter', 'Int', 'A float']
)

display(df)

In [ ]:
print("The index of the dataframe:")
display(df.index)

### Exercise 1
Create a DataFrame called `df` that contains the data in the following table:

|       | **A** | **B** |
|-------|-------|-------|
| **X** | 5.3   | foo   |
| **Y** | 2.0   | bar   |
| **Z** | 0.2   | baz   |

In [ ]:
df = None

# YOUR CODE HERE

display(df)

In [ ]:
test_1(df)

*Further Reading:* You can find the full documentation on creating a DataFrame [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html#pandas.DataFrame) (which lists the technical description of all the options) or use the Pandas User Guide [on DataFrames](https://pandas.pydata.org/pandas-docs/stable/user_guide/dsintro.html#basics-dataframe) (which includes more examples).

Now that we know how to create a DataFrame, let's see what we can do with them!

This notebook will follow a narrative where you are hired as the new head of (and only member of) the Data Processing department of your local grocery store. This grocery store has kept a record of their products, and it is now up to you to process this data. This record consists of products, to what category they belong, their price, the size (i.e. 'unit') of the product and the current stock.

We have provided you with a function `get_df()` which gets you the data as a `DataFrame` for you to use. As you can see, it is _indexed_ on the 'Product' and has the _columns_ 'Category', 'Price', 'Unit' and 'Stock'. The example `DataFrame` is displayed below.

In [ ]:
df = get_df()
display(df)

### Getting data from a DataFrame

A table consists of columns with values for each row; and similarly, a `DataFrame` consists of `Series` with values for each index. Therefore, if you want to look at a single column of your data, you'll need to retrieve that `Series` from your `DataFrame`!

To see what columns are in your DataFrame, you can use the `df.columns` property:

In [ ]:
print("The columns of `df`:")
display(df.columns)

# Indexing works the same way as a list:
print("\nThe second column is:", df.columns[1])

To get a column from your DataFrame, you can index it in the same way you do with a dictionary: by using `df['ColumnName']`, where `df` is your DataFrame and `ColumnName` is the name of your column. The code below retrieves the 'Price' column from our DataFrame:

In [ ]:
# Retrieve the 'Price' column from the DataFrame as a Series
prices = df['Price']

display(prices)

In [ ]:
# Here you can see that this is indeed a 'Series' object:
print("The variable 'prices' is of type:", type(prices))

# You can always check the dtype of your Series by using .dtype:
print("This Series has dtype:", prices.dtype)

In the example above you can see that this Series doesn't just contain the _values_ of the 'Price' column (displayed on the right), but also the _index_, which lists the product names from the original DataFrame (displayed on the left). This way, it is a lot easier to see which price belongs to what product! Lastly, you can see at the bottom what the `dtype` of this column is (in this case `float64`, since these are floating point numbers).

Accessing the data in that `Series` then also works in the same way as for dictionaries: you can just use the index as the key. For example, the price of Broccoli can be found by first retrieving the 'Price' column from the `DataFrame`, and then indexing the resulting `Series` on 'Broccoli':

In [ ]:
# Retrieve the 'Price' column from the DataFrame as a Series
prices = df['Price']

# Get the 'Broccoli' value from that Series
price_broccoli = prices['Broccoli']

print("The price of Broccoli is:", price_broccoli)

### Exercise 2

In the 'Stock' column, how many 'Penne' are left? Use `df` to find the value and save the result in `stock_penne`.

In [ ]:
df = get_df()
stock_penne = None

# YOUR CODE HERE

print(f"There are {stock_penne} penne in stock.")

In [ ]:
test_2(stock_penne)

### Series

A `Series` is a data structure specific to *Pandas*, that has aspects of both lists and dictionaries. It is like a `list` in that it is an ordered sequence of data, meaning it just stores different values together in some order. However, the indices used for a `Series` don't need to be the integers $0, 1, \dots$, like with a list, but can be any type, including strings, like the keys of a `dictionary`.

Note that all the values in a `Series` must be of the same type. In the example above, the `Series` has a `dtype` of float, meaning it can only contain floating point values. Lastly, a `Series` has an optional name, which is set automatically here to the name of the column from the `DataFrame` where it came from.

If we want to create a `Series`, we can just define it using a list of indices and list of values. Note that order here matters; position 0 in the index list will become the index for position 0 in the value list, and this will become the first element in the `Series`. Below is an example to create a `pd.Series` called `discount` with products as its indices and a discounted price as its values.

In [ ]:
discount_products = ['Strawberry', 'Apple', 'Coffee', 'Juice', 'Broccoli']
discount_prices = [2.99, 2.15, 4.99, 1.89, 0.99]

discount = pd.Series(
    data=discount_prices,      # Use discount_prices as the values
    index=discount_products,   # Use discount_products as the index
    dtype=float,               # Set the type for the `data` argument to floating point numbers
    name="Discount",           # Set the name for the Series to Discount
)

display(discount)


The most basic version of a `Series` is just a sequence of data of the same type, meaning you actually don't _need_ to provide an index for your `Series`. If you do not provide any index, `Pandas` will just use $0, 1, \dots$, like the indices of a `list`.

However, much of the power of Pandas is the use of this index to easily retrieve rows from your DataFrame (more on this in the section on [Indexing, Selection and Masking](#indexing-selection-and-masking)). It is therefore always recommended to use _some_ sort of index for your Series, if you can!

Below is an example of the basic way you can create a Series, by just providing a `list` of values:

In [ ]:
basicSeries = pd.Series([6.4, 2.3, 1.0])

display(basicSeries)

You can create a `Series` with any kind of data. Below are some more examples with different data types: pay attention to the resulting `dtype` for each!

Pandas tries its best to find the right type for your Series. A Series must always be consistent in its type for all values in the Series, and the standard types are: integers (`int64`), floats (`float64`), booleans (`bool`) and strings (`str`).

When pandas *cannot* find a single type that fits all the values — say you mix strings and numbers in one Series — it falls back on the generic `object` type. That is pandas telling you "I gave up: these are just Python objects to me". Seeing `object` in a column is nearly always a signal that something needs cleaning.

What you can do with a column largely depends on the `dtype` of that `Series`, so always try to pick the type that works for your data!

In [ ]:
print("A Series with strings gets dtype 'str':")
display(pd.Series(['a', 'b', 'c']))

print("\nIf you provide a mix of integers and floats, it will convert all elements to floats:")
display(pd.Series([1, 2, 3.0]))

print("\nBut you can also force it to be a specific type:")
display(pd.Series([1, 2, 3.0], dtype=int))

print("\nIf you provide a mix of strings and other types, then everything becomes 'object's:")
display(pd.Series(['hello', 2, 3.0]))

print("\nAs a very useful extra, Pandas can even convert your dates to a consistent type:")
display(pd.Series([ "2023 aug 02", "2021 8 23", "1970/01/01"], dtype="datetime64[ns]"))

### Exercise 3

At the end of the day, the following sales come in:

|            |   |
|------------|---|
| Lettuce    | 1 |
| Carrot     | 3 |
| Spaghetti  | 1 | 
| Coffee     | 2 |
| Apple      | 1 |
| Strawberry | 1 |

Create a `Series` called `sales` containing the data in the table above.

In [ ]:
sales = None

# YOUR CODE HERE

display(sales)

In [ ]:
test_3(sales)

### Adding and modifying data

You can *add* a column to your `DataFrame` in the same way we retrieved a column from the DataFrame before. If you have a Series called `mySeries`, you can add it to your DataFrame `myDataFrame` as a column named `'myColumn'` using `myDataFrame['myColumn'] = mySeries`. Pandas will even match the `index` of your Series and DataFrame, so all values end up in the right place!

Let's use the `discount` Series from before and add it to the DataFrame. This will match all the indices in the Series with the DataFrame indices, and add the values in the right place. When there is no discount for a specific product, the value automatically gets set to `NaN`. We will come back to the different ways you can deal with the `NaN`s later.

In [ ]:
df = get_df()

print("The discount Series")
discount = pd.Series(discount_prices, index=discount_products)
display(discount)

print("Added to the DataFrame:")
df['Discount'] = discount
display(df)

We can also easily modify values in `Series` using basic arithmetic operations. All arithmetic operations are element-wise by default, and broadcasted when necessary.

Below is a simple example where we get a new inventory shipment with 10 items for each product, and need to update the DataFrame accordingly. Note that this increase is getting broadcasted to every value in the `Series` here.

It is important to remember that any arithmetic operation always creates a **new** `Series`, and so doesn't modify the existing DataFrame! If we want to update the values in the DataFrame, we need to re-assign the new Series to the corresponding column.

In [ ]:
print("Adding 10 to the stock of every product:")
increased_stock = df['Stock'] + 10
display(increased_stock)

print("\nReplacing the stock column in the DataFrame:")
df['Stock'] = increased_stock
display(df)

An important detail to note is that element-wise operations are always matched by *their index* (so in this case, the product name: "Apple", "Banana", etc.), so the order of the elements in the Series actually does not matter, only what their index is. We'll come back to this in more detail later, but for now we'll just give another example, so you have a bit of an idea of the possibilities:

Let's say you now want to compute the percentage of discount for each of the discounted prices, so you can use those to advertise your sale. First we'll need to compute the ratio of the discounted prices compared to the original price and store the result in a new `Series`. We can just do this by dividing the Discount column by the Price column:

In [ ]:
discount_ratio = df['Discount'] / df['Price']

print("The ratio of the discounted price compared to the original price:")
display(discount_ratio)

Note that this division was indeed performed element by element and matched on the index, so the discounted price of an apple was divided by the original price of an apple, and so on. The products that had a `NaN` value for the discount, then also automatically become `NaN` for the ratio.

Next, we can just multiply this by $100$ to get a percentage, and subtract that value from $100$ to get the percentage that was discounted. Lastly, we can then add this new `Series` back into the `DataFrame` as a new column:

In [ ]:
discount_percentage = 100 - discount_ratio * 100
df['Percentage'] = discount_percentage

print('Discount percentage added to the DataFrame:')
display(df)

In the `DataFrame` we can now see the apples have $10\%$ discount when sold for $2.15$ instead of $2.39$, but we might want to advertise with the $33\%$ discount on coffee instead!

With a couple of discounts this is easy to spot, but for a larger `DataFrame` you would use built-in *aggregate* functions instead. Aggregate functions combine data in a column in some way, so for example using a `.sum()` to compute the total for a column. Here we'll need to use the `.max()` function to find the largest discount percentage and `.idxmax()` to find the product that this largest discount belongs to. Note that `NaN` values in the Percentage column will just be ignored here, as this is the default behaviour for all aggregation functions.

In [ ]:
largest_discount_percentage = df['Percentage'].max()
largest_discount_product = df['Percentage'].idxmax()

original_price = df['Price'][largest_discount_product]
discount_price = df['Discount'][largest_discount_product]
product_unit = df['Unit'][largest_discount_product]

print(f'Our best deal is a {int(largest_discount_percentage)}% discount on {largest_discount_product}:')
print(f'Usually {original_price} per {product_unit}, but now on sale for {discount_price}!')

### Exercise 4

In Exercise 3 we created a `sales` Series with the amount sold, and the prices of each product are listed in the product DataFrame, so we can now see how much revenue we made from these products!

Create a new Series called `revenue`, which combines the number of items sold from the `sales` Series with the prices from the product DataFrame `df`. The resulting Series should indicate the revenue per product and be added to the DataFrame as a column called Revenue. Finally, compute the total revenue by aggregating all the values in the `revenue` Series and store this as `total_revenue`.

In [ ]:
print("As a quick reminder, this is the data in the sales Series:")
display(sales)

In [ ]:
# Reset the DataFrame
df = get_df()

revenue = None
total_revenue = None

# YOUR CODE HERE

print("Revenue per item, added as new column in the DataFrame:")
display(df)

print(f"\n\nIn total, the market had a revenue of: ${total_revenue:.2f}")

In [ ]:
test_4(revenue, total_revenue)

*Further Reading:* The official documentation has more about [creating a Series](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.html#pandas.Series). Pandas User Guide also has a [section on Series](https://pandas.pydata.org/pandas-docs/stable/user_guide/dsintro.html#basics-series) and the [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html) introduces Pandas using Series.

## Chapter 2: Reading files & sanity checking

Normally when you're working with data you will not manually create the DataFrame, but read them from some file. Pandas can read data from many different file types. In this chapter we will read data from comma-separated values (.csv) files.

### The data

We'll now have a look at a completely different dataset. We're going to use two synthetic files from the same seabird monitoring programme, two years apart:
- **`survey_2022.csv`**: the **published** dataset. Curated and checked before release.
- **`survey_2023.csv`**: the **raw export**, straight back from the field. Nobody has checked it.

> Disclaimer! **This data is synthetic.** Both files were generated for teaching purposes only. 

We'll inspect the published file first, to learn what *healthy* data looks like.

#### Data dictionary (codebook)

| column | type | expected values |
|---|---|---|
| `date` | text (YYYY-MM-DD) | survey date, April–July |
| `site` | text | one of: `Cliff East`, `Dune North`, `Dune South`, `Harbour` |
| `species` | text | one of: `Atlantic Puffin`, `Razorbill`, `Common Guillemot` |
| `sex` | text | `M` or `F` |
| `body_mass_g` | number | 300–1200 g |
| `wing_length_mm` | number | 150–340 mm |
| `bill_length_mm` | number | 25–55 mm |
| `observer` | text | `Team 1`, `Team 2` or `Team 3` |

Each file holds one survey season of 200 records. Measurements that could not be taken
in the field are recorded as `-999`.

Real datasets often come with a codebook like this.

### Read the data

We use `pd.read_csv()` for reading files. There are a lot of parameters we can configure when reading files, but for this particular file we do not need any of them: the defaults work fine.

In [ ]:
survey = pd.read_csv("survey_2022.csv")

#### Validation step 1: inspect

Once we have loaded a file, we need to validate the data. Is everything the way we would expect? With very small DataFrames you can still inspect the data visually, but in reality we often work with very large datasets, where visual inspection is no longer feasible.

First just have a look at the first few rows, to see if nothing obvious went wrong:

In [ ]:
survey.head()

#### Validation step 2: shape and types

`shape` tells you whether you got the number of records you expected (the codebook says 200). `info()` shows the dtypes and the non-null counts in one go.

Watch the **dtypes**: a column of measurements must be a number (`int64`/`float64`). If it shows up as `str` (or `object`), pandas found something in there that isn't a number. This could become a problem later as every calculation on that column will misbehave.

In [ ]:
print(survey.shape)

survey.info()

#### Validation step 3: plausible values?

`describe()` gives you the numeric summary. Compare `min` and `max` against the expected ranges in the codebook. This is where impossible values give themselves away.

In [ ]:
survey.describe()

#### Validation step 4: are the categories what they should be?

`value_counts()` on each text column. You're checking two things: that the categories are **exactly** the ones in the codebook, and that the counts look sensible.

In [ ]:
for column in ["site", "species", "sex", "observer"]:
    print(survey[column].value_counts())
    print()

#### Validation step 5: what's missing?

`isna().sum()` counts the missing values per column. If everything is well, they should all be `0`, meaning that there are no missing values.

In [ ]:
survey.isna().sum()

#### What healthy data looks like

Every check came back reassuring, and that is exactly the point:

| check | result |
|---|---|
| `head()` | sensible rows, no junk |
| `shape` | 200 records, as documented |
| `dtypes` | all three measurements numeric |
| `describe()` | every range inside the codebook's bounds |
| `value_counts()` | exactly 4 sites, 3 species, 2 sexes, 3 teams |
| `isna().sum()` | all zeros |

Boring is the goal. It took thirty seconds, and now you can have some faith in the data going forward. Keep in mind, though, that these are just basic sanity checks. There are many things that could still be wrong that we haven't checked yet. We'll come back to that later.

The important lesson is that you always validate data (when you read it from a file, but also after every transformation you do on it yourself). You inspect it **every time**, whether you suspect a problem or not!

### Exercise 5

The file **`survey_2023.csv`** covers the same programme, with the same columns and the
same codebook — but it came straight out of the field and nobody has checked it. It has
problems. Several of them.

Load it yourself and run the same five steps on it to find out what's wrong.

Describe at least 5 issues with this data. There are more than five to
find, so keep going once you have five — every extra one you spot is worth having.

For each issue, report three things:

| # | What is wrong | How you found it | What it would break |
|---|---|---|---|
| 1 | e.g. *`sex` has six different codings* | `raw["sex"].value_counts()` | counting males vs females would be wrong |

**Do not fix anything yet.** Right now the skill you're practising is *noticing* (which is often harder than the fixing).

In [ ]:
# Load the file and run the five inspection steps.

# YOUR CODE HERE


#### Your data health report

Describe **at least 5** issues you found, in the table format above.

YOUR ANSWER HERE

Even a thorough sanity check does not catch everything.

You may have noticed the **row count doesn't match the codebook** (203 vs 200). Now you know *something* is off, but not *what* (the cause is duplicated records, and you need `duplicated()` to prove it).

This file contains problems that are **completely invisible** to all five steps. For example, a few birds are recorded with a body mass that is perfectly normal *for the colony* but impossible *for their species*. Every value sits inside the codebook's range, so no single-column check can see it. Catching that means comparing one column **against another**, or comparing each row **against its group**.

In the following notebooks we will also discuss how you can run validations for those issues.

In fact, you'll get a first glimpse of exactly this in Chapter 4: a scatter plot compares two columns at once, and that is enough to make one of these hidden problems visible.

## Chapter 3: Selecting & filtering

### Indexing, Selection and Masking

In the first chapter we covered the most basic ways to retrieve data from a `DataFrame`, but there are many more ways to get data from a `DataFrame` in *Pandas*. As we've already covered how to retrieve columns, next we'll take a look at how to get specific rows from a `DataFrame`.

The first thing we need to answer is: What is a 'row' in *Pandas*? Turns out: it's also a `Series`! That's great, because we just learned how to work with those. Let's start by taking another look at the example `DataFrame` from the first chapter:

In [ ]:
df = pd.read_csv("products.csv", index_col=0)

display(df)

(Note that now that we know how to read data from files, we don't have to use `get_df()` anymore to retrieve the DataFrame.)

There are several ways to access the rows in a `DataFrame`, but the most common ones are `df.loc[...]` and `df.iloc[...]`. These two methods look and work very similarly, but there are some important differences:

### Using `df.loc[...]`

The `.loc` property retrieves rows by making use of the *index* we defined for our dataframe. This works similarly to a dictionary, where you can use the key (i.e. index) to get the row you want: `df.loc['Banana']` will get you the row where the index is equal to 'Banana'.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
row = df.loc['Banana']

print("The row with index 'Banana' is:")
display(row)
print("\nThis row is of the type:", type(row))

As you can see, the resulting row `Series` automatically becomes indexed by the *columns* from the original DataFrame. This means that, for example, you could now retrieve the 'Price' for a specific `row` using `row['Price']`. This works the same way for any row, regardless of whether you're using either `.loc` or `.iloc` to retrieve it.

### Using `df.iloc[...]`

The `.iloc` property retrieves rows by the order in which the rows are stored in the `DataFrame`. It stands for "integer location" and works the same way as `list` indices: `df.iloc[0]` is the first row, `df.iloc[1]` is the second, et cetera.

It is important to note that if you re-order your dataframe (such as with sorting), the order of the rows will change and therefore also their "Integer location"! After some operations, using `df.iloc[0]` might give a different result than before. This is different from `df.loc`, where you just get the row matching the index regardless of the exact position in the `DataFrame`.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
row = df.iloc[0]

print("The first row of the DataFrame is:")
display(row)
print("\nThis row is of the type:", type(row))

### Rows vs. columns

Retrieving specific rows from a `DataFrame` can obviously be very useful, but there is an important caveat to keep in mind when working with rows:

The problem here stems from the fact that a `Series` can only ever have one type. This works great for columns, as you expect a column of data to all be of the same type (e.g. a price column should always contain floats for each row). This is probably also the best way to look at what a `DataFrame` actually is: A collection of different columns of data, each with their *own data type*, that also each share the *same row index*.

However, a single row of data will contain information from all of the different columns, which might have different types. For mixed types, *Pandas* will again try its best to convert the types to a consistent option, but if you have just one column with strings, then the whole row `Series` will become type 'object'. You can actually already see this happening in both the examples above!

Converting everything to 'object' type means you lose the type information from the columns. In addition, these rows are actually *new* `Series` that have to be built by retrieving the values from each column for an index, and then combining them together into that `Series` (which is when the type conversion happens). This is **much** slower than getting a column, where you can simply retrieve the stored `Series` from a `DataFrame`.

As a result, the best option is usually to work with columns directly. Looping over the rows in a `DataFrame` might seem like a natural way to process data, as you're going through the data line by line, but this is actually very inefficient. Instead you should try and use element-wise operations to modify a whole column in a `DataFrame`, like the examples you've already seen in the section [Adding and modifying data](#Adding-and-modifying-data).

Note that it is still possible to loop through the rows of a `DataFrame` using functions like [`.iterrows()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html). This function will create a new `Series` for each row and return them one by one, which can sometimes be very useful. However, if you do find yourself using this, ask yourself first if there isn't a better way to solve your problem, as this type of approach should always be one of the last options you try.

### Exercise 6

Select the fifth row from `df` using the `.iloc[...]` property and store it in the variable `fifth`.

Next, select the row about Lettuce from `df` and save it in the variable `lettuce` using the `.loc[...]` property.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
fifth = None
lettuce = None

# YOUR CODE HERE

print("Fifth row:")
display(fifth)

print("Lettuce:")
display(lettuce)

In [ ]:
test_6(fifth, lettuce)

### Selecting multiple rows or columns

In addition to using `.loc['Banana']` or `.iloc[8]` to obtain a single row, you can also pass a `list` of values to obtain multiple rows from your DataFrame in one go! This actually also works for *columns*, by passing a list of column names when indexing the DataFrame. Note that all these operations will *always* return a `DataFrame`, and not a `Series` like before — even if it is a `DataFrame` with only a single row or column!

Below are some examples using lists to index a DataFrame:

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
print("The full dataframe:")
display(df)

rows1 = df.loc[ ['Carrot', 'Juice', 'Rice'] ]
print("\nThe rows with index 'Carrot', 'Juice' and 'Rice':")
display(rows1)

rows2 = df.loc[ ['Carrot'] ]
print("\nThe row with index 'Carrot', as a DataFrame:")
display(rows2)

rows3 = df.iloc[ [0, 2, 4] ]
print("\nFirst, third and fifth row, using a list of indices for iloc:")
display(rows3)

rows4 = df[ ['Category', 'Price'] ]
print("\nSelecting only the Category and Price columns")
display(rows4)

The `df.iloc` property works with the position index, and also supports 'slices':
- `df.iloc[a:]` returns everything starting from the `a`-th row.
- `df.iloc[:b]` returns everything up to (but not including) the `b`-th row.
- `df.iloc[::c]` returns every `c`-th row, starting with the first.
    You can also think of this as the 'step size' with which to go through the DataFrame. By default this is 1, which is just every row. A step size of 2 means you skip every second row, et cetera.

And these can also be combined as `df.iloc[a:b:c]` which says: from the `a`th to the `b`th row, give every `c`th row starting with the `a`th.

*Further Reading:* You can find more examples on slices [here](https://www.geeksforgeeks.org/python-list-slicing/).

Below are some examples. Feel free to try out some different values in a new cell, and predict what you'll see before running the code.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
print("Full DataFrame, with a new default range as index:")
# If you reset the index, it will use indices 0,1,... instead
# This will help see what integer indexes are for each row!
display(df.reset_index())

print("\n\nStarting at the tenth row:")
display(df.iloc[10:])

print("\n\nFirst five rows:")
display(df.iloc[:5])

# A negative index means: count backwards from the end
print("\n\nStarting at the second-to-last row:")
display(df.iloc[-2:])

# A bit of an advanced example:
# 1:10 says "Take the second to tenth rows (indices 1,2,3,4,5,6,7,8,9)"
#   :2 says "And then skip every other row (indices 1,3,5,7,9)"
print("\n\nEvery second row up to the tenth row:")
display(df.iloc[1:10:2])

### Exercise 7
Create a variable `groceries` which contains all the information in `df` for Broccoli, Coffee and Rice using the `.loc` property.

Create a variable `odds` which contains all the odd rows (first, third, fifth, et cetera) using `.iloc` and slices.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
groceries = None
odds = None

# YOUR CODE HERE

print("Groceries:")
display(groceries)

print("\n\nOdd rows:")
display(odds)

In [ ]:
test_7(groceries, odds)

### Combining row and column selection

If you want to select specific rows *and* columns from a `DataFrame`, there are many ways you can do this in *Pandas*. First off, you can just combine the methods for selecting rows and columns you saw earlier, as these always return a `DataFrame` which you can just index again.

Some examples:

In [ ]:
df = pd.read_csv("products.csv", index_col=0)

print("\nSelecting columns first, then indexing the resulting rows using loc:")
cols = df[['Category', 'Price']]
res1 = cols.loc[['Carrot', 'Juice', 'Rice']]
display(res1)


print("\nIndexing the rows using their position with iloc, then selecting the columns:")
rows = df.iloc[[3, 6, 9]]
res2 = rows[['Category', 'Price']]
display(res2)


print("\nSelecting columns first, then slices on iloc, executed on the same line:")
res3 = df[['Category', 'Price']].iloc[3:10:3]
display(res3)

Note that all these different methods end up with the same DataFrame! Actually any combination of row and column indexing you've seen before would work for this, of course.

In addition, `loc` and `iloc` also support the option to index rows and columns together in the same operation, which is a little more efficient than first creating new Series for the full rows and then selecting from them. With these methods the row indices must always come first (as `loc` and `iloc` are row-based operations), and then optionally you can add a column index too, separated by a comma. This combined indexing also works for single indices, where you only give one index for the rows or columns position, instead of a list. Note in those cases the row or column is reduced, and the result will no longer be a DataFrame. A few examples:

In [ ]:
df = pd.read_csv("products.csv", index_col=0)

print("\nSelecting the rows with index Apple and Broccoli, and the columns Category and Stock")
display(df.loc[ ['Apple', 'Broccoli'], ['Category', 'Stock'] ])

print("\nSelecting the first and second row, and the second and fourth column:")
display(df.iloc[ [0, 1], [1, 3] ])

print("\nSelecting the Penne row and the Price column (which is just an element):")
display(df.loc['Penne', 'Price'])

print("\nSelecting only the first row, and the first and fourth column (which is a Series):")
display(df.iloc[0, [0, 3]])


All these different indexing methods can make *Pandas* especially confusing to newcomers, as they are all used interchangeably in explanations and examples you might find online. One of the goals of this notebook is to introduce all of these properly, so you can make a bit more sense of the different options you might encounter.

Below are 3 examples all using `loc` that look very similar, but do completely different things. Read the code of the examples and try to predict what each of these will do, before actually running the cell:

In [ ]:
example1 = df.loc['Apple', 'Price']

print("\nExample 1:")
display(example1)

example2 = df.loc[['Apple', 'Broccoli']]

print("\nExample 2:")
display(example2)

example3 = df.loc[['Apple'], ['Price']]

print("\nExample 3:")
display(example3)


### Exercise 8

Below are 3 exercises to test your understanding with:

a) Select the Broccoli, Coffee and Rice rows from the DataFrame, while also only selecting the Category and Price columns. Save the results in `ex_8a`.

b) Select the last 3 rows from the DataFrame, while also only selecting the Price and Unit columns. Save the results in `ex_8b`.

c) Select the Unit and Stock for Juice. The results should be Series indexed with Unit and Stock. Save the results in `ex_8c`.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
ex_8a = None
ex_8b = None
ex_8c = None

# YOUR CODE HERE

print("\n8a:")
display(ex_8a)

print("\n8b:")
display(ex_8b)

print("\n8c:")
display(ex_8c)

In [ ]:
test_8(ex_8a, ex_8b, ex_8c)

### Masking

The last, and very useful, method we'll cover to select data from your DataFrame is masking.

Masks are used to select parts of a DataFrame where some condition is `True`. You can create a mask by using boolean operators (like `>`, `<=` or `==`) directly on a DataFrame or Series to define the condition where the mask should be `True`. This will result in a broadcasted comparison of elements on the DataFrame/Series, so will create a new DataFrame/Series containing only `True`s and `False`s. Applying a mask just takes the part(s) of your DataFrame where the mask is `True`, and leaves out the parts where it is `False`.

The code below uses the Stock column to create a mask called `mask` where the Stock column is less than four, then uses it to show what items in the DataFrame are low in stock. Note that the `dtype` of the mask becomes `bool`ean and it indeed contains only `True` for the rows that have a Stock less than 4.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)

mask = df['Stock'] < 4

print("\nMask where Stock is less than 4:")
display(mask)

print("\nUsing the mask on the DataFrame gives:")
display(df[mask])

Pandas also has built-in functions to create masks. One such example is the `pd.Series.isin(values)` method. This mask is `True` whenever an element in a Series is equal to one of the elements you provided as the *list* of `values`.

As an example, the code below creates a mask where the values in the 'Category' column are either 'Vegetables' or 'Fruits'.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)

mask = df['Category'].isin(['Vegetables', 'Fruits'])

print("\nThe mask where the category is either Fruits or Vegetables:")
display(mask)

print("\n\nThe result of applying this mask on the DataFrame:")
display(df[mask])

You can also use logical operations, like *logical or* `|` and the *logical and* `&`, to combine two masks element-wise. The first creates a new mask where each element is `True` if one of the masks was `True` for that element, and the second creates a new mask where each element is `True` if both the masks were `True` for that element.

In [ ]:
expensive = df['Price'] > 1
low_stock = df['Stock'] < 4

combined_mask = expensive & low_stock

print("\nThe combined mask for items that are both expensive and low in stock:")
display(combined_mask)

print("\nSelecting out the rows where the combined mask is True:")
display(df[combined_mask])

### Exercise 9

Select the rows from the DataFrame where the Category is either Pastas or Fruits, and the Price is either below 1 or above 3. Store the result as `ex_9`.

In [ ]:
df = pd.read_csv("products.csv", index_col=0)
ex_9 = None

# YOUR CODE HERE

display(ex_9)

In [ ]:
test_9(ex_9)

### Validation: comparing two columns

Comparing columns can often also be useful whilst validating data. Every check you ran in the previous chapter looked at one column at a time: the range
of `bill_length_mm`, the categories in `species`, the missing values in `wing_length_mm`.

But some rules only exist *between* columns. Masks are a natural way to express those,
because you can compare two columns directly: the comparison happens element-wise, row by
row, and gives you back a mask just like any other condition.

Here is a rule that relies on common sense: **a bird's bill cannot be
longer than its wing.** You can only validate this in the data by comparing the `bill_length_mm` and `wing_length_mm` columns.

In [ ]:
raw = pd.read_csv("survey_2023.csv")

impossible = raw[raw["bill_length_mm"] > raw["wing_length_mm"]]

print(f"{len(impossible)} record(s) where the bill is longer than the wing:")
display(impossible[["species", "wing_length_mm", "bill_length_mm"]])

One record turns up: a Common Guillemot with a wing length of `0` and a bill of `50 mm`.

Now, that particular row would also have been caught by `describe()`, because a wing length
of zero is impossible on its own. But it shows you yet another way to validate data, which can be useful in the future.

*Further Reading:* The Python Data Science Handbook has a section on [data indexing and selection](https://jakevdp.github.io/PythonDataScienceHandbook/03.02-data-indexing-and-selection.html) and using [masks](https://jakevdp.github.io/PythonDataScienceHandbook/02.06-boolean-arrays-and-masks.html). The documentation on [`.iloc`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iloc.html) and [`.loc`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.loc.html) also contains more examples.

## Chapter 4: Basic visualization with pandas

So far you have inspected your data by reading numbers: counts, means, minima and maxima.
That works, but sometimes its much easier to see patterns visually than in a
table of numbers. A single plot can show you the shape of a whole column at once.

So plotting is two things at once: a way to present results, and one of the fastest validation tools you have.

We will focus on the second use here. A histogram exposes an impossible value immediately,
and a bar chart of categories exposes a misspelled one. You saw both of these problems in
the previous chapter by reading output.

Pandas has plotting built in through the `.plot` accessor, which uses the `matplotlib`
library under the hood. You call it directly on a `Series` or a `DataFrame`, and you use
`matplotlib` itself only for the finishing touches (titles, labels).

In [ ]:
import matplotlib.pyplot as plt

# We'll use the clean survey data from the previous chapter.
survey = pd.read_csv("survey_2022.csv")

survey.head()

### Bar charts: comparing categories

A bar chart is a good choice when you want to compare a value across a handful of
categories. The most common case is counting how often each category occurs, which you
already know how to do with `value_counts()`. You can plot the result directly:

In [ ]:
counts = survey["species"].value_counts()

counts.plot(kind="bar")

plt.title("Number of birds measured per species")
plt.xlabel("Species")
plt.ylabel("Number of records")
plt.show()

Note that `value_counts()` produces a `Series` whose *index* is the category
and whose *values* are the counts, and `.plot(kind="bar")` turns exactly that into a chart.
The index becomes the labels along the bottom, the values become the bar heights.

When the category names are long, a *horizontal* bar chart (`kind="barh"`) is usually
easier to read, because the labels get a whole line each instead of being squeezed
underneath the bars.

### Histograms: the shape of one numeric column

A histogram answers "how are the values in this column distributed?". It splits the range
of the column into equal-width *bins* and counts how many values fall into each. Use
`bins=` to control how fine the picture is: too few bins hides structure, too many turns
it into noise.

This can be a useful plot for checking a numeric column.

In [ ]:
survey["body_mass_g"].plot(kind="hist", bins=30)

plt.title("Distribution of body mass")
plt.xlabel("Body mass (g)")
plt.ylabel("Number of birds")
plt.show()

Look carefully at that shape: it has more than one peak. That is not an error, it gives useful information about our data: Our survey contains three species with quite different
body masses, so what you are seeing is three overlapping distributions together.

### Scatter plots: the relationship between two numeric columns

Where a histogram shows you one column, a scatter plot shows you two at once: each row
becomes a point, with one column on the x-axis and the other on the y-axis. This is how
you check whether two measurements are related.

For a scatter plot you call `.plot` on the DataFrame (not on a single column), because
you need to name two columns. The `alpha` argument makes the points slightly transparent,
which helps when many of them overlap.

In [ ]:
survey.plot(kind="scatter", x="wing_length_mm", y="body_mass_g", alpha=0.6)

plt.title("Body mass against wing length")
plt.xlabel("Wing length (mm)")
plt.ylabel("Body mass (g)")
plt.show()

Again the picture shows more than a single cloud: the points fall into separate clumps,
one per species. A scatter plot is often the quickest way to notice that your data has
structure you weren't accounting for.

### Plotting as validation

In the previous chapter you found the problems in `survey_2023.csv`
by reading output carefully. Let's see how quickly the same problems announce themselves
in a picture.

First, a histogram of the wing lengths from the raw export:

In [ ]:
raw = pd.read_csv("survey_2023.csv")

raw["wing_length_mm"].plot(kind="hist", bins=30)

plt.title("Wing length in the raw export")
plt.xlabel("Wing length (mm)")
plt.ylabel("Number of birds")
plt.show()

There is essentially nothing to see: one tall bar on the left and a vast empty space to
the right. A single impossible value of `9999` stretches the x-axis
so far that all the real measurements are squashed into one bin. An outlier that would take
a moment to notice in `describe()` becomes unmissable here.

We can use a similar trick for categories. Plotting the species counts shows immediately that
three species have turned into six bars:

In [ ]:
raw["species"].value_counts().plot(kind="barh")

plt.title("Species in the raw export")
plt.xlabel("Number of records")
plt.show()

### The one thing a plot will not tell you

Plots are excellent validators with one important blind spot: missing values are silently
left out. A histogram of a column that is half empty looks perfectly healthy (nothing in
the picture tells you how many rows never made it into the chart at all).

So a plot can never replace `isna().sum()`. Compare the two numbers yourself:

In [ ]:
print("rows in the file:              ", len(raw))
print("wing lengths actually plotted: ", raw["wing_length_mm"].notna().sum())
print("silently left out:             ", raw["wing_length_mm"].isna().sum())

Twelve birds simply were not drawn, and the histogram gave you no hint of it. This is why
the checks from the previous chapter and the plots from this one are complementary: use
the numbers to establish what is there, and the pictures to see its shape.

### Two columns at once

Every check in the previous chapter looked at one column at a time: the values in
`bill_length_mm`, the categories in `species`, the missing values in `wing_length_mm`. A
scatter plot is the first tool you've met that looks at two columns together, and that
turns out to catch a whole class of problem the others cannot.

Let's plot bill length against wing length for the raw export. Straight from the file, it
is unreadable (the `-999` bills and the `9999` wing length stretch both axes so far that
every real bird collapses into a corner):

In [ ]:
raw.plot(kind="scatter", x="wing_length_mm", y="bill_length_mm", alpha=0.6)

plt.title("Bill against wing length, straight from the raw export")
plt.xlabel("Wing length (mm)")
plt.ylabel("Bill length (mm)")
plt.show()

So first exclude the values you already know are impossible. The codebook says wing
lengths run 150–340 mm and bill lengths 25–55 mm, so anything outside that is not a
measurement. This is a good habit in itself: you often have to remove the obvious problems
before you can see the more subtle issue.

In [ ]:
plausible = raw[(raw["wing_length_mm"] >= 150) & (raw["wing_length_mm"] <= 340)
                & (raw["bill_length_mm"] >= 25) & (raw["bill_length_mm"] <= 55)]

print(f"{len(plausible)} of {len(raw)} records have both measurements in range")

plausible.plot(kind="scatter", x="wing_length_mm", y="bill_length_mm", alpha=0.6)

plt.title("Bill against wing length, plausible records only")
plt.xlabel("Wing length (mm)")
plt.ylabel("Bill length (mm)")
plt.show()

Now three clusters appear, one per species, exactly as you'd hope. Puffins have short
wings and long bills (top left), Razorbills have long wings and short bills (bottom
right), Guillemots have long wings and long bills (top right).

So here is the interesting question: if each species occupies its own region, what happens
if we plot only the birds recorded as one species? They should form a single tidy clump.

In [ ]:
puffins = plausible[plausible["species"] == "Atlantic Puffin"]

print(f"{len(puffins)} records are labelled 'Atlantic Puffin'")

puffins.plot(kind="scatter", x="wing_length_mm", y="bill_length_mm", alpha=0.7)

plt.title("Records labelled 'Atlantic Puffin'")
plt.xlabel("Wing length (mm)")
plt.ylabel("Bill length (mm)")
plt.show()

They don't. There is one tidy clump of 71 birds where puffins belong (wing 156–185 mm,
bill 40–51 mm), and then two lonely points far off to the right, at wing lengths of
about 202 and 215 mm with bills of only 36 mm. That is not what a puffin normally looks like but it
is precisely where the Razorbills live.

Those two records are probably wrong. Either the species was written down incorrectly, or the
measurements belong to a different bird. We cannot tell which from the data alone, but we
can say with some confidence that something about those two rows is inconsistent.

Now notice what makes this problem different from everything you found in the previous
chapter. Look at those two rows one column at a time and they are entirely innocent:

- their `species` is a valid, correctly-spelled category,
- their `wing_length_mm` is a perfectly normal wing length,
- their `bill_length_mm` is a perfectly normal bill length.

Every single-column check passes. The problem only exists in the combination (a wing
length that is fine on its own but impossible *for the species it is paired with*). No
amount of `describe()` or `value_counts()` will ever reveal it, because those functions
never look at two columns at the same time.

This is a reason that simple sanity checks are not sufficient. In the
notebooks that follow you'll learn to express checks like this as code rather than
eyeballing a chart: comparing one column against another, and comparing each row against
the group it belongs to.

> One caveat, and a nice illustration of how errors compound: we filtered with
> `species == "Atlantic Puffin"`, so the records spelled `atlantic puffin` and
> `Altantic Puffin` were silently left out of this plot entirely. Before you could trust a
> check like this, you would first have to fix the spelling problem from the previous
> chapter. Data problems rarely queue up politely (fixing one is often a prerequisite for
> even *seeing* the next).

### Exercise 10

Make a horizontal bar chart of the price of every product in the grocery data
(`products.csv`), sorted from cheapest to most expensive.

Give the chart a title and label the axis that shows the price (including its unit).

In [ ]:
products = pd.read_csv("products.csv", index_col=0)

# YOUR CODE HERE

### Exercise 11

Pick one of the problems you found in `survey_2023.csv` in the previous chapter, and
make a plot that puts it on display. Anything other than the wing-length outlier and the
species spellings (we did those above) is fair game.

Then, in the cell below the plot, write one sentence explaining what your plot reveals.

In [ ]:
# YOUR CODE HERE

**What does your plot reveal?**

YOUR ANSWER HERE